# 03. Spatio-Temporal Incident Correlation & Lag Anomaly Detection

Computes trend + day-of-week sales residuals, attaches geo-fenced incident exposure with Haversine distance decay, and estimates distributed lag impacts (`t`, `t+1`, `t+2`).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/bga-matplotlib")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from src.config import load_config
from src.osint_collector import load_incidents_file
from src.spatial_models import estimate_lag_effects, join_sales_with_incidents

DATA_MODE = os.environ.get("BGA_DATA_MODE", "sample").strip().lower()
cfg = load_config(repo_root=REPO_ROOT)
if not cfg.coords_verified:
    print("NOTICE: Using approximate Silao coords (20.95, -101.42) with coords_verified=False.")

In [ ]:
if DATA_MODE == "raw":
    from src.db_loader import load_daily_sales
    daily_df = load_daily_sales(config=cfg)
else:
    daily_df = pd.read_csv(cfg.sample_dir / "daily_sales.csv", parse_dates=["date"])

incidents_df = load_incidents_file(cfg.sample_dir / "incidents.csv", relevant_only=True)
joined = join_sales_with_incidents(
    sales_df=daily_df,
    incidents_df=incidents_df,
    store_lat=cfg.lat,
    store_lon=cfg.lon,
    radii_m=cfg.geofence_radii_m,
    max_distance_m=float(max(cfg.geofence_radii_m)),
    lags=(0, 1, 2),
)

lag_results = estimate_lag_effects(joined, outcome_col="residual_pct", lags=(0, 1, 2))
print("Daily Distributed Lag Regression (t, t+1, t+2):")
print(lag_results.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(joined["date"], joined["residual_pct"] * 100.0, alpha=0.7, label="Sales Residual (%)")
inc_hits = joined[joined["has_incident_lag0"]]
axes[0].scatter(inc_hits["date"], inc_hits["residual_pct"] * 100.0, color="#d95f02", zorder=4, label="Incident Day (t=0)")
axes[0].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[0].set_title("Daily Revenue Residuals vs OSINT Incident Days")
axes[0].legend()

axes[1].errorbar(
    lag_results["lag"],
    lag_results["coef"] * 100.0,
    yerr=(lag_results["ci_high"] - lag_results["coef"]) * 100.0,
    fmt="o-",
    capsize=4,
)
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(["t (Same Day)", "t+1", "t+2"])
axes[1].set_title("Estimated Incident Exposure Effect (%) with 95% CI")
plt.tight_layout()
plt.close(fig)